# IIDS67692 Computational Techniques for Multi-modal Data
# WSI-Bench GPT-2 + LoRA with Perturbation-Aware Semantic Hallucination Entropy

This notebook adapts the PathVQA PA-SHE experiment to **WSI-Bench**. It uses
the WSI patch-feature files referenced by the official annotations rather than
trying to load a gigapixel slide as one ordinary image.

In [ ]:
!nvidia-smi

### Dependency environment

In [ ]:
# Dependencies are managed in ~/dissertation_2026/.venv before submission.
# Never install or upgrade packages inside an nbconvert job sharing that environment.
import sys
print("Using preconfigured Python environment:", sys.executable)

# Dataset: WSI-Bench annotations and local WSI patch features

Before running this cell:

1. Accept access to `Lucas-yuc/WSI-Bench` on Hugging Face and authenticate with
   `huggingface-cli login` (or export `HF_TOKEN`).
2. Download the complete separate `Lucas-yuc/WSIBench-pt` feature repository (or use
   a complete existing copy) and point `WSIBENCH_FEATURE_ROOT` to its root. Check the
   download size first with `hf download Lucas-yuc/WSIBench-pt --repo-type dataset
   --local-dir /path/to/features --dry-run`, then remove `--dry-run` to download.
   The annotation `image`
   values must resolve to `.pt`, `.pth`, `.npy`, `.npz`, `.h5`, or `.hdf5`
   feature files below that root.
3. Optionally set `WSIBENCH_ANNOTATION_DIR`; otherwise annotations are stored
   under `data/WSI-Bench/annotations` in the current project.

The public release has a training annotation file and an open-test file; a
local closed-test file is used when available. It has no validation file, so this notebook makes
a deterministic **slide-level** validation split from training data and never
uses the official test questions for training or model selection.

In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

WSIBENCH_REPO_ID = "Lucas-yuc/WSI-Bench"
WSIBENCH_FEATURE_REPO_ID = "Lucas-yuc/WSIBench-pt"
WSIBENCH_ANNOTATION_DIR = Path(
    os.environ.get("WSIBENCH_ANNOTATION_DIR", "data/WSI-Bench/annotations")
).expanduser().resolve()
WSIBENCH_FEATURE_EXTENSIONS = {
    ".pt", ".pth", ".npy", ".npz", ".h5", ".hdf5"
}


def wsibench_find_feature_root():
    """Resolve an explicit or common project-local patch-feature root."""
    requested = os.environ.get("WSIBENCH_FEATURE_ROOT", "").strip()
    project_dir = Path.cwd().resolve()
    candidates = []
    if requested:
        candidates.append(Path(requested).expanduser())
    candidates.extend([
        WSIBENCH_ANNOTATION_DIR.parent / "features",
        WSIBENCH_ANNOTATION_DIR / "features",
        project_dir / "data" / "WSI-Bench" / "features",
        project_dir / "WSI-Bench" / "features",
        Path.home() / "dissertation_2026" / "data" / "WSI-Bench" / "features",
        Path.home() / "dissertation_2026" / "joint_se_wsibench" / "data" / "WSI-Bench" / "features",
    ])
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.is_dir() and any(
            path.is_file() and path.suffix.lower() in WSIBENCH_FEATURE_EXTENSIONS
            for path in candidate.rglob("*")
        ):
            return candidate

    checked_text = "\n  - ".join(str(path) for path in checked)
    requested_text = requested or "<not set>"
    raise FileNotFoundError(
        "WSI-Bench requires precomputed patch-feature files; the annotation "
        "download does not create them.\n"
        f"WSIBENCH_FEATURE_ROOT={requested_text!r}\n"
        f"Checked:\n  - {checked_text}\n"
        "Set the real directory before nbconvert, for example:\n"
        "  export WSIBENCH_FEATURE_ROOT=\"${HOME}/dissertation_2026/"
        "joint_se_wsibench/data/WSI-Bench/features\"\n"
        "Do not leave the placeholder /path/to/your/wsi/features."
    )


WSIBENCH_FEATURE_ROOT = wsibench_find_feature_root()
WSIBENCH_VALIDATION_FRACTION = float(
    os.environ.get("WSIBENCH_VALIDATION_FRACTION", "0.10")
)
WSIBENCH_SPLIT_SEED = int(os.environ.get("WSIBENCH_SPLIT_SEED", "42"))
WSIBENCH_TEST_MODE = os.environ.get("WSIBENCH_TEST_MODE", "both").strip().lower()

if not 0.0 < WSIBENCH_VALIDATION_FRACTION < 1.0:
    raise ValueError("WSIBENCH_VALIDATION_FRACTION must be between 0 and 1.")
if WSIBENCH_TEST_MODE not in {"both", "open", "closed"}:
    raise ValueError("WSIBENCH_TEST_MODE must be: both, open, or closed.")
WSIBENCH_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
WSIBENCH_FILES = {
    "train": "WSI-Bench-train.json",
    "open": "WSI-Bench-open-question.jsonl",
}
WSIBENCH_CLOSED_FILENAME = "WSI-Bench-close-question.jsonl"


def wsibench_annotation_path(filename, required=True):
    local_path = WSIBENCH_ANNOTATION_DIR / filename
    if local_path.exists():
        return local_path
    try:
        downloaded = hf_hub_download(
            repo_id=WSIBENCH_REPO_ID,
            repo_type="dataset",
            filename=filename,
            local_dir=WSIBENCH_ANNOTATION_DIR,
            token=os.environ.get("HF_TOKEN") or None,
        )
    except Exception as error:
        if not required:
            return None
        raise RuntimeError(
            f"Could not obtain gated WSI-Bench annotation {filename}. "
            "Accept the dataset conditions and run `huggingface-cli login`, "
            "or copy the file into WSIBENCH_ANNOTATION_DIR."
        ) from error
    return Path(downloaded)


def wsibench_read_records(path):
    path = Path(path)
    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as handle:
            rows = [json.loads(line) for line in handle if line.strip()]
        return [
            record
            for row in rows
            for record in (row if isinstance(row, list) else [row])
        ]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("data", "records", "annotations", "items"):
            if isinstance(payload.get(key), list):
                return payload[key]
    raise ValueError(f"Unsupported annotation structure: {path}")


def wsibench_first(record, keys, default=None):
    for key in keys:
        value = record.get(key)
        if value is not None and str(value).strip():
            return value
    return default


def wsibench_scalar_text(value, preferred_keys=()):
    if value is None:
        return ""
    if isinstance(value, dict):
        for key in (*preferred_keys, "text", "value", "content", "answer"):
            if key in value:
                text = wsibench_scalar_text(value[key], preferred_keys)
                if text:
                    return text
        return ""
    if isinstance(value, (list, tuple)):
        for item in value:
            text = wsibench_scalar_text(item, preferred_keys)
            if text:
                return text
        return ""
    return str(value).strip()


def wsibench_clean_question(value):
    text = str(value or "")
    text = re.sub(r"<image(?:_placeholder)?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def wsibench_conversation_pairs(record):
    conversation = wsibench_first(
        record, ("conversations", "conversation", "messages"), []
    )
    if not isinstance(conversation, list):
        return []
    pairs = []
    pending_question = None
    for message in conversation:
        if not isinstance(message, dict):
            continue
        role = str(wsibench_first(message, ("from", "role", "speaker"), "")).lower()
        value = wsibench_first(message, ("value", "content", "text"), "")
        if role in {"human", "user", "question"}:
            pending_question = wsibench_clean_question(value)
        elif role in {"gpt", "assistant", "answer"} and pending_question:
            pairs.append((pending_question, str(value).strip()))
            pending_question = None
    return pairs


def wsibench_infer_answer_type(question, answer, source_kind, record):
    if source_kind == "open":
        return "Open-ended"
    if source_kind == "closed":
        return "Closed-ended"
    declared = str(
        wsibench_first(record, ("answer_type", "question_type", "type", "mode"), "")
    ).lower()
    closed_markers = ("close", "multiple", "choice", "mcq", "true", "false", "yes/no")
    if any(marker in declared for marker in closed_markers):
        return "Closed-ended"
    question_lower = str(question).lower()
    answer_lower = str(answer).strip().lower()
    has_options = bool(re.search(r"(?:^|\s)[a-d][\).:]\s", question_lower))
    if has_options or answer_lower in {"yes", "no", "true", "false", "a", "b", "c", "d"}:
        return "Closed-ended"
    return "Open-ended"


def wsibench_normalize_records(raw_records, source_kind):
    normalized = []
    for raw_index, record in enumerate(raw_records):
        if not isinstance(record, dict):
            continue
        image_ref = wsibench_scalar_text(wsibench_first(
            record,
            (
                "image", "image_id", "image_name", "image_path",
                "feature", "feature_path", "wsi", "wsi_id",
                "wsi_name", "wsi_path", "slide", "slide_id",
                "slide_name", "slide_path",
            ),
            "",
        ), ("path", "file_name", "filename", "id"))
        slide_id = str(wsibench_first(
            record,
            ("slide_id", "slide", "wsi_id", "case_id"),
            Path(image_ref).stem or f"record-{raw_index}",
        )).strip()
        direct_question = wsibench_scalar_text(wsibench_first(
            record,
            ("question", "questions", "prompt", "instruction",
             "query", "text", "text_input", "input"),
            None,
        ), ("question", "prompt"))
        direct_answer = wsibench_scalar_text(wsibench_first(
            record,
            ("answer", "answers", "T-answer", "T_answer",
             "t-answer", "t_answer", "reference", "reference_answer",
             "response", "label", "gt_answer", "ground_truth",
             "gt", "text_output", "output", "target"),
            None,
        ), ("answer", "text"))
        pairs = wsibench_conversation_pairs(record)
        if direct_question and direct_answer:
            pairs.insert(0, (
                wsibench_clean_question(direct_question),
                direct_answer,
            ))
        for pair_index, (question, answer) in enumerate(pairs):
            if not image_ref or not question or not answer:
                continue
            normalized.append({
                "record_id": str(wsibench_first(
                    record, ("id", "question_id", "uid"),
                    f"{source_kind}-{raw_index}-{pair_index}",
                )),
                "image": image_ref,
                "slide_id": slide_id,
                "question": question,
                "answer": answer,
                "answer_type": wsibench_infer_answer_type(
                    question, answer, source_kind, record
                ),
                "source_file": source_kind,
            })
    if not normalized:
        sample_types = sorted({type(record).__name__ for record in raw_records[:20]})
        sample_keys = sorted({
            str(key)
            for record in raw_records[:20] if isinstance(record, dict)
            for key in record.keys()
        })
        raise ValueError(
            f"No usable question/answer records were parsed from {source_kind}. "
            f"Sample record types={sample_types}; keys={sample_keys}."
        )
    return normalized


annotation_paths = {
    kind: wsibench_annotation_path(filename, required=True)
    for kind, filename in WSIBENCH_FILES.items()
}
annotation_paths["closed"] = wsibench_annotation_path(
    WSIBENCH_CLOSED_FILENAME, required=False
)
wsibench_train_all = wsibench_normalize_records(
    wsibench_read_records(annotation_paths["train"]), "train"
)
wsibench_open_test = wsibench_normalize_records(
    wsibench_read_records(annotation_paths["open"]), "open"
)
if annotation_paths["closed"] is None:
    wsibench_closed_test = []
    warnings.warn(
        "Closed-test annotations were not found in the release; "
        "WSIBENCH_TEST_MODE='both' will evaluate the available open test."
    )
else:
    wsibench_closed_test = wsibench_normalize_records(
        wsibench_read_records(annotation_paths["closed"]), "closed"
    )

if WSIBENCH_TEST_MODE == "open":
    wsibench_test = wsibench_open_test
elif WSIBENCH_TEST_MODE == "closed":
    if not wsibench_closed_test:
        raise FileNotFoundError(
            "WSIBENCH_TEST_MODE='closed' requires a local "
            f"{WSIBENCH_CLOSED_FILENAME} in {WSIBENCH_ANNOTATION_DIR}."
        )
    wsibench_test = wsibench_closed_test
else:
    wsibench_test = wsibench_open_test + wsibench_closed_test

# Remove any official-test slide from the training pool, then split remaining
# slides rather than individual questions to prevent slide leakage.
test_slide_ids = {record["slide_id"] for record in wsibench_test}
development_pool = [
    record for record in wsibench_train_all
    if record["slide_id"] not in test_slide_ids
]
development_slide_ids = sorted({record["slide_id"] for record in development_pool})
rng = np.random.default_rng(WSIBENCH_SPLIT_SEED)
rng.shuffle(development_slide_ids)
validation_slide_count = max(
    1, int(round(len(development_slide_ids) * WSIBENCH_VALIDATION_FRACTION))
)
validation_slide_ids = set(development_slide_ids[:validation_slide_count])
wsibench_validation = [
    record for record in development_pool
    if record["slide_id"] in validation_slide_ids
]
wsibench_train = [
    record for record in development_pool
    if record["slide_id"] not in validation_slide_ids
]


def wsibench_debug_limit(records, environment_name):
    limit = int(os.environ.get(environment_name, "0"))
    return records if limit <= 0 else records[:limit]


wsibench_train = wsibench_debug_limit(wsibench_train, "WSIBENCH_MAX_TRAIN")
wsibench_validation = wsibench_debug_limit(
    wsibench_validation, "WSIBENCH_MAX_VALIDATION"
)
wsibench_test = wsibench_debug_limit(wsibench_test, "WSIBENCH_MAX_TEST")

split_summary = pd.DataFrame([
    {
        "split": name,
        "questions": len(records),
        "slides": len({record["slide_id"] for record in records}),
        "open_questions": sum(record["answer_type"] == "Open-ended" for record in records),
        "closed_questions": sum(record["answer_type"] == "Closed-ended" for record in records),
    }
    for name, records in {
        "train": wsibench_train,
        "validation": wsibench_validation,
        "official test": wsibench_test,
    }.items()
])
display(split_summary)

train_slides = {record["slide_id"] for record in wsibench_train}
validation_slides = {record["slide_id"] for record in wsibench_validation}
assert train_slides.isdisjoint(validation_slides)
assert train_slides.isdisjoint(test_slide_ids)
assert validation_slides.isdisjoint(test_slide_ids)
print("Annotation paths:", {key: str(value) for key, value in annotation_paths.items()})
print("WSI feature root:", WSIBENCH_FEATURE_ROOT)
print("Test mode:", WSIBENCH_TEST_MODE)

# Prepare WSI feature dataloaders

In [ ]:
from collections import OrderedDict
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

WSIBENCH_MAX_PATCHES = int(os.environ.get("WSIBENCH_MAX_PATCHES", "256"))
WSIBENCH_EXPECTED_FEATURE_DIM = int(
    os.environ.get("WSIBENCH_FEATURE_DIM", "0")
)
WSIBENCH_FEATURE_CACHE_SIZE = int(
    os.environ.get("WSIBENCH_FEATURE_CACHE_SIZE", "32")
)
if WSIBENCH_MAX_PATCHES < 1:
    raise ValueError("WSIBENCH_MAX_PATCHES must be positive.")

WSIBENCH_FEATURE_EXTENSIONS = {".pt", ".pth", ".npy", ".npz", ".h5", ".hdf5"}
WSIBENCH_FEATURE_INDEX = {}
for feature_path in WSIBENCH_FEATURE_ROOT.rglob("*"):
    if feature_path.is_file() and feature_path.suffix.lower() in WSIBENCH_FEATURE_EXTENSIONS:
        WSIBENCH_FEATURE_INDEX.setdefault(feature_path.name, feature_path)
        WSIBENCH_FEATURE_INDEX.setdefault(feature_path.stem, feature_path)
WSIBENCH_INDEXED_FEATURE_COUNT = len(
    {str(path) for path in WSIBENCH_FEATURE_INDEX.values()}
)
print("Indexed WSI feature files:", WSIBENCH_INDEXED_FEATURE_COUNT)


def wsibench_resolve_feature_path(image_reference):
    reference = Path(str(image_reference))
    candidates = [WSIBENCH_FEATURE_ROOT / reference, WSIBENCH_FEATURE_ROOT / reference.name]
    if reference.suffix.lower() not in WSIBENCH_FEATURE_EXTENSIONS:
        candidates.extend(
            WSIBENCH_FEATURE_ROOT / f"{reference.name}{extension}"
            for extension in sorted(WSIBENCH_FEATURE_EXTENSIONS)
        )
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    for key in (reference.name, reference.stem):
        if key in WSIBENCH_FEATURE_INDEX:
            return WSIBENCH_FEATURE_INDEX[key]
    raise FileNotFoundError(
        f"Could not resolve WSI feature '{image_reference}' below {WSIBENCH_FEATURE_ROOT}."
    )


def wsibench_tensor_from_payload(payload):
    if torch.is_tensor(payload):
        return payload
    if isinstance(payload, np.ndarray):
        return torch.from_numpy(payload)
    if isinstance(payload, dict):
        preferred = ("features", "feature", "embeddings", "embedding", "x", "data")
        for key in preferred:
            if key in payload:
                return wsibench_tensor_from_payload(payload[key])
        for value in payload.values():
            try:
                tensor = wsibench_tensor_from_payload(value)
                if tensor.ndim >= 2:
                    return tensor
            except (TypeError, ValueError):
                pass
    if isinstance(payload, (list, tuple)):
        for value in payload:
            try:
                tensor = wsibench_tensor_from_payload(value)
                if tensor.ndim >= 2:
                    return tensor
            except (TypeError, ValueError):
                pass
    raise TypeError("No numeric patch-feature tensor was found in the feature payload.")


def wsibench_load_feature_file(path):
    suffix = path.suffix.lower()
    if suffix in {".pt", ".pth"}:
        try:
            payload = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            payload = torch.load(path, map_location="cpu")
    elif suffix == ".npy":
        payload = np.load(path, allow_pickle=False)
    elif suffix == ".npz":
        archive = np.load(path, allow_pickle=False)
        payload = archive[archive.files[0]]
    elif suffix in {".h5", ".hdf5"}:
        with h5py.File(path, "r") as handle:
            datasets = []
            handle.visititems(
                lambda name, obj: datasets.append(np.asarray(obj))
                if isinstance(obj, h5py.Dataset) and obj.ndim >= 2 else None
            )
        if not datasets:
            raise ValueError(f"No 2-D feature dataset found in {path}")
        payload = datasets[0]
    else:
        raise ValueError(f"Unsupported feature extension: {path.suffix}")

    tensor = wsibench_tensor_from_payload(payload).detach().cpu().float().squeeze()
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)
    elif tensor.ndim > 2:
        tensor = tensor.reshape(-1, tensor.shape[-1])
    if tensor.ndim != 2:
        raise ValueError(f"Expected a 2-D patch-feature tensor in {path}; got {tensor.shape}")

    common_dimensions = {256, 384, 512, 768, 1024, 1280, 1536, 2048, 2560}
    if (
        WSIBENCH_EXPECTED_FEATURE_DIM > 0
        and tensor.shape[0] == WSIBENCH_EXPECTED_FEATURE_DIM
        and tensor.shape[1] != WSIBENCH_EXPECTED_FEATURE_DIM
    ):
        tensor = tensor.transpose(0, 1)
    elif tensor.shape[1] not in common_dimensions and tensor.shape[0] in common_dimensions:
        tensor = tensor.transpose(0, 1)
    tensor = torch.nan_to_num(tensor, nan=0.0, posinf=0.0, neginf=0.0).contiguous()
    if tensor.shape[0] < 1 or tensor.shape[1] < 1:
        raise ValueError(f"Empty feature tensor in {path}")
    return tensor


class WSIBenchDataset(Dataset):
    def __init__(self, records, feature_dim=None, max_patches=WSIBENCH_MAX_PATCHES):
        self.dataset = list(records)
        self.max_patches = int(max_patches)
        self._feature_cache = OrderedDict()
        first_path = wsibench_resolve_feature_path(self.dataset[0]["image"])
        first_tensor = wsibench_load_feature_file(first_path)
        inferred_dim = int(first_tensor.shape[1])
        self.feature_dim = inferred_dim if feature_dim is None else int(feature_dim)
        if inferred_dim != self.feature_dim:
            raise ValueError(
                f"Feature dimension mismatch: expected {self.feature_dim}, got {inferred_dim}"
            )

    def __len__(self):
        return len(self.dataset)

    def _load(self, image_reference):
        path = wsibench_resolve_feature_path(image_reference)
        cache_key = str(path)
        if cache_key in self._feature_cache:
            tensor = self._feature_cache.pop(cache_key)
            self._feature_cache[cache_key] = tensor
            return tensor
        tensor = wsibench_load_feature_file(path)
        if tensor.shape[1] != self.feature_dim:
            raise ValueError(
                f"Inconsistent feature dimension in {path}: "
                f"expected {self.feature_dim}, got {tensor.shape[1]}"
            )
        if WSIBENCH_FEATURE_CACHE_SIZE > 0:
            self._feature_cache[cache_key] = tensor
            while len(self._feature_cache) > WSIBENCH_FEATURE_CACHE_SIZE:
                self._feature_cache.popitem(last=False)
        return tensor

    def __getitem__(self, index):
        sample = self.dataset[index]
        features = self._load(sample["image"])
        # Deterministic uniform token selection. If a slide has fewer tokens,
        # linspace repeats valid tokens rather than introducing padded zeros.
        token_indices = torch.linspace(
            0, features.shape[0] - 1, steps=self.max_patches
        ).round().long()
        features = features[token_indices].clone()
        return features, str(sample["question"]), str(sample["answer"])


# Fail early with a compact missing-file report rather than during an epoch.
required_feature_references = sorted({
    str(record["image"])
    for record in (wsibench_train + wsibench_validation + wsibench_test)
})
missing_feature_references = []
for image_reference in required_feature_references:
    try:
        wsibench_resolve_feature_path(image_reference)
    except FileNotFoundError:
        missing_feature_references.append(image_reference)
        if len(missing_feature_references) >= 20:
            break
if missing_feature_references:
    raise FileNotFoundError(
        "The WSI feature collection is incomplete for the selected official splits.\n"
        f"Only {WSIBENCH_INDEXED_FEATURE_COUNT} feature files were indexed below "
        f"{WSIBENCH_FEATURE_ROOT}.\n"
        "First missing annotation references:\n  - "
        + "\n  - ".join(missing_feature_references)
        + "\nComplete the feature collection before rerunning. From the shell, first check "
        + "the download size and then download into the same root:\n"
        + f"  hf download {WSIBENCH_FEATURE_REPO_ID} --repo-type dataset "
        + f"--local-dir \"{WSIBENCH_FEATURE_ROOT}\" --dry-run\n"
        + f"  hf download {WSIBENCH_FEATURE_REPO_ID} --repo-type dataset "
        + f"--local-dir \"{WSIBENCH_FEATURE_ROOT}\"\n"
        + "Do not filter out missing slides for the final evaluation."
    )

train_data = wsibench_train
val_data = wsibench_validation
test_data = wsibench_test

requested_feature_dim = (
    WSIBENCH_EXPECTED_FEATURE_DIM if WSIBENCH_EXPECTED_FEATURE_DIM > 0 else None
)
train_dataset = WSIBenchDataset(train_data, feature_dim=requested_feature_dim)
WSIBENCH_FEATURE_DIM = train_dataset.feature_dim
val_dataset = WSIBenchDataset(val_data, feature_dim=WSIBENCH_FEATURE_DIM)
test_dataset = WSIBenchDataset(test_data, feature_dim=WSIBENCH_FEATURE_DIM)

print({
    "train_questions": len(train_dataset),
    "validation_questions": len(val_dataset),
    "official_test_questions": len(test_dataset),
    "feature_dimension": WSIBENCH_FEATURE_DIM,
    "tokens_per_slide": WSIBENCH_MAX_PATCHES,
})

features, question, answer = train_dataset[0]
plt.figure(figsize=(10, 4))
plt.imshow(features[:64, :128], aspect="auto", cmap="viridis")
plt.colorbar(label="feature value")
plt.xlabel("feature dimension (first 128)")
plt.ylabel("WSI patch token (first 64)")
plt.title(f"Q: {question[:100]}\nA: {answer[:100]}", fontsize=10)
plt.tight_layout()
plt.show()

#Model Architecture

Paper: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

GPT-2 uses a decoder-only transformer architecture with multiple model sizes; the commonly used GPT-2 Base model contains 12 transformer blocks (layers), a context window of 1024 tokens, a hidden embedding size of 768, and about 117 million parameters, while larger variants scale up to 48 transformer blocks and 1.5 billion parameters.

[1] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. OpenAI blog, 1(8), 9.

###Cross-Attention Fusion

In [ ]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model

####Cross-Attention Fusion###########
class CrossAttentionFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, text_embeds, image_embeds, text_att_mask=None):
        """
        text_embeds:  [B, T, 768]
        image_embeds: [B, N, 768]
        text attends to image
        """

        attended_text, attn_weights = self.cross_attn(
            query=text_embeds,
            key=image_embeds,
            value=image_embeds,
            need_weights=False
        )

        x = self.norm1(text_embeds + attended_text)
        x = self.norm2(x + self.ffn(x))

        return x

### Gated WSI feature compression

In [ ]:
# The trainable feature gate is implemented in WSIFeatureEncoder below.

### WSI feature encoder + multimodal GPT-2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from peft import get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class WSIFeatureEncoder(nn.Module):
    """Project and compress precomputed patch features into visual tokens."""

    def __init__(self, input_dim, hidden_dim=768, output_tokens=32):
        super().__init__()
        self.output_tokens = int(output_tokens)
        self.projection = nn.Linear(int(input_dim), hidden_dim, bias=False)
        self.normalization = nn.LayerNorm(hidden_dim)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        if features.ndim != 3:
            raise ValueError(
                f"Expected WSI features [batch, patches, dimension]; got {features.shape}"
            )
        tokens = self.normalization(self.projection(features))
        tokens = tokens * self.gate(tokens)
        tokens = F.adaptive_avg_pool1d(
            tokens.transpose(1, 2), self.output_tokens
        ).transpose(1, 2)
        return tokens


class MedVQA(nn.Module):
    def __init__(self, peft_config=None):
        super().__init__()
        output_tokens = int(os.environ.get("WSIBENCH_VISUAL_TOKENS", "32"))
        self.visual_encoder = WSIFeatureEncoder(
            input_dim=WSIBENCH_FEATURE_DIM,
            hidden_dim=768,
            output_tokens=output_tokens,
        )

        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        gpt = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt = get_peft_model(gpt, peft_config)
        self.fusion = CrossAttentionFusion(hidden_dim=768, num_heads=4)

    def forward(self, image, qa_inputs_ids, qa_att_mask):
        image_embeds = self.visual_encoder(image)
        text_embeds = self.gpt.get_input_embeddings()(qa_inputs_ids)
        fused_embeds = self.fusion(
            text_embeds=text_embeds,
            image_embeds=image_embeds,
            text_att_mask=qa_att_mask,
        )
        return self.gpt(
            inputs_embeds=fused_embeds,
            attention_mask=qa_att_mask,
        ).logits

# Model training or checkpoint reuse

This notebook reuses its own WSI-Bench checkpoint by default:

`checkpoints_wsibench_pa_she/best_model_ca_wsibench_pa_she.pth`

If the file is absent, training runs for up to **10 epochs** with validation
early stopping (patience **5**) and saves the best model. Set
`REUSE_TRAINED_CHECKPOINT=0` to deliberately retrain. This checkpoint is
separate from the PathVQA and VQA-RAD checkpoints because the visual encoder
and feature dimension are WSI-Bench-specific.

In [ ]:
#Training Script for Multimodal GPT2 with LoRA
import os
import torch
import argparse
import torch.utils.data
import numpy as np
import random

from torch import nn
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

import evaluate
from nltk.translate.bleu_score import corpus_bleu
from peft import  TaskType, LoraConfig

import warnings
warnings.filterwarnings('ignore')

REUSE_TRAINED_CHECKPOINT = os.environ.get(
    'REUSE_TRAINED_CHECKPOINT', '1'
).strip().lower() not in {'0', 'false', 'no'}


def load_vqa_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def adjust_learning_rate(optimizer, shrink_factor):
    print("\nDECAYING learning rate.")
    for param_group in optimizer.param_groups:
        param_group['lr'] = param_group['lr'] * shrink_factor
    print("The new learning rate is %f\n" % (optimizer.param_groups[0]['lr'],))

def train(args, train_dataloader, model, criterion, optimizer, epoch, tokenizer, device):
    model.train()
    total_loss = []

    for i, (images, questions, answers) in enumerate(train_dataloader, 0):
        # prepare prompts
        qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
        qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

        # get labels
        labels = qa_prompt_inputs['input_ids'].clone()
        labels = labels.to(device)

        # for labels, mask question tokens and padding tokens
        for idx, q in enumerate(questions):
            q_prompt = f"Question: {q}\nAnswer: "
            q_length = len(tokenizer(q_prompt)["input_ids"]) - 1

            labels[idx, :q_length] = -100  # mask question
            eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
            if eos_mask.sum() > 1:  # if more than 1 EOS
                first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS

        # get logits and labels
        logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
        )

        # get shifted logits and labels
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        # compute loss
        shift_logits = shift_logits.view(-1, shift_logits.size(-1))
        shift_labels = shift_labels.view(-1)
        loss = criterion(shift_logits, shift_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss.append(loss.item())
        if i%50 == 0:
            print("Training - Epoch: {}/{}, Iteration: {}/{}, Training Loss: {:.6f}".format(epoch, args.epochs, i, len(train_dataloader), np.array(total_loss).mean()))


def validate(args, val_loader, model, criterion, epoch, tokenizer, device):
    total_loss = []
    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(val_loader, 0):
            # prepare prompts
            qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
            qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

            # get labels
            labels = qa_prompt_inputs['input_ids'].clone()
            labels = labels.to(device)

            # for labels, mask question tokens and padding tokens
            answer_starts = []
            answer_ends = []
            for idx, q in enumerate(questions):
                q_prompt = f"Question: {q}\nAnswer: "
                q_length = len(tokenizer(q_prompt)["input_ids"]) - 1
                answer_starts.append(q_length+1)

                labels[idx, :q_length] = -100  # mask question
                eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
                if eos_mask.sum() > 1:  # if more than 1 EOS
                    first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                    labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS
                    answer_ends.append(first_eos_pos)

            # get logits and labels
            logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
            )

            # get shifted logits and labels
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            # compute loss
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            loss = criterion(shift_logits, shift_labels)
            total_loss.append(loss.item())

    return np.array(total_loss).mean()


def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


def get_arg():
    parser = argparse.ArgumentParser(description='VisualQuestionAnswerGeneration')
    # Training parameters
    parser.add_argument('--epochs',         type=int,   default=10,   help='number of epochs to train for')
    parser.add_argument('--batch_size',     type=int,   default=32,   help='training and validation batch size')
    parser.add_argument('--workers',        type=int,   default=8,    help='for data-loading')
    parser.add_argument('--random_seed',    type=int,   default=42,   help='random seed')
    parser.add_argument('--seq_length',     type=int,   default=160,  help='sequence length for question and answer')
    parser.add_argument('--dropout', type=float, default=0.1, help='dropout')
    parser.add_argument('--early_stopping_patience', type=int, default=5,
                        help='stop after this many epochs without validation improvement')

    parser.add_argument('--dataset',        default='endo',  help='endo / pit')
    parser.add_argument('--lr',             type=float, default=0.0002,  help='0.0000001, 0.00000005')
    parser.add_argument('--checkpoint_dir', default='checkpoints_wsibench_pa_she/',
                        help='separate checkpoint path for the PA-SHE experiment')

    args = parser.parse_args([])
    return args


if __name__ == '__main__':

    args = get_arg()
    seed_everything(args.random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f'Batch size: {args.batch_size}')
    print(f'Learning rate: {args.lr}')
    print(f'Random seed: {args.random_seed}')
    print(f'Sequence length: {args.seq_length}')
    print(f'Maximum epochs: {args.epochs}')
    print(f'Early-stopping patience: {args.early_stopping_patience}')

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    MODEL_CHECKPOINT_PATH = os.path.join(
        args.checkpoint_dir,
        'best_model_ca_wsibench_pa_she.pth',
    )
    reuse_checkpoint = (
        REUSE_TRAINED_CHECKPOINT
        and os.path.isfile(MODEL_CHECKPOINT_PATH)
    )
    checkpoint_saved_this_run = False
    start_epoch = 1
    epochs_since_improvement = 0
    best_val_loss = float('inf')

    print('Dataset: WSI-Bench slide-disjoint train/validation splits')
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )

    print(
        'DataLoader configuration:',
        {
            'train_examples': len(train_dataset),
            'validation_examples': len(val_dataset),
            'batch_size': args.batch_size,
            'workers': args.workers,
        },
    )

    # init tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    model = model.to(device)

    # for name, param in model.named_parameters():
    #     if param.requires_grad:
    #         print(name)

    pytorch_total_params = sum(p.numel() for p in model.parameters())
    print('model params: ', pytorch_total_params)

    # init optimizer and criterion
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100).to(device)

    # Reuse this notebook's validation-selected checkpoint unless retraining
    # is explicitly requested or the checkpoint does not exist.
    if reuse_checkpoint:
        print(
            'Reusing trained checkpoint; skipping epoch training:',
            MODEL_CHECKPOINT_PATH,
        )
        training_epochs = []
    else:
        if REUSE_TRAINED_CHECKPOINT:
            print(
                'No existing checkpoint found; training from scratch:',
                MODEL_CHECKPOINT_PATH,
            )
        else:
            print('Checkpoint reuse disabled; training from scratch.')
        print('Start training.')
        training_epochs = range(start_epoch, args.epochs + 1)

    for epoch in training_epochs:
        if epochs_since_improvement > 0 and epochs_since_improvement % 5 == 0:
            adjust_learning_rate(optimizer, 0.8)

        # train
        train(args, train_dataloader=train_dataloader, model=model, criterion=criterion, optimizer=optimizer,
              epoch=epoch, tokenizer=tokenizer, device=device)
        # validation
        val_loss = validate(args, val_loader=val_dataloader, model=model, criterion=criterion,
                            epoch=epoch, tokenizer=tokenizer, device=device)

        if val_loss < best_val_loss:  # save model with better validation loss
            epochs_since_improvement = 0
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_CHECKPOINT_PATH)
            checkpoint_saved_this_run = True
            model.tokenizer.save_pretrained(args.checkpoint_dir)
            print('Best validation loss, model saved.')
        else:
            epochs_since_improvement += 1
            print("\nEpochs since last improvement: %d\n" % (epochs_since_improvement,))

        if epochs_since_improvement >= args.early_stopping_patience:
            print(
                f'Early stopping at epoch {epoch}: validation loss did not improve '
                f'for {args.early_stopping_patience} consecutive epochs.'
            )
            break

    if not reuse_checkpoint:
        if not checkpoint_saved_this_run:
            raise RuntimeError(
                'Training finished without producing a validation checkpoint.'
            )
        print(f'End training. Best validation loss: {best_val_loss:.6f}')

    if not os.path.isfile(MODEL_CHECKPOINT_PATH):
        raise FileNotFoundError(
            f'Model checkpoint not found: {MODEL_CHECKPOINT_PATH}'
        )
    model.load_state_dict(
        load_vqa_checkpoint(MODEL_CHECKPOINT_PATH, device)
    )
    model.to(device)
    model.eval()
    print('Loaded validation-selected checkpoint:', MODEL_CHECKPOINT_PATH)

# Validation inference: qualitative WSI-feature sanity check

In [ ]:
# Validation-only prediction visualisations (official test remains untouched)
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


def greedy_search_single(image, question, model, tokenizer, max_length, device):
    model.eval()
    with torch.no_grad():
        prompt = tokenizer(
            f"Question: {question}\nAnswer:",
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = prompt["input_ids"].to(device)
        attention = prompt["attention_mask"].to(device)
        generated = []
        image = image.unsqueeze(0).to(device)
        for _ in range(max(0, max_length - input_ids.shape[1])):
            logits = model(
                image=image,
                qa_inputs_ids=input_ids,
                qa_att_mask=attention,
            )[0, -1]
            next_id = torch.argmax(logits)
            if int(next_id) == tokenizer.eos_token_id:
                break
            generated.append(int(next_id))
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
            attention = torch.cat([
                attention,
                torch.ones((1, 1), dtype=attention.dtype, device=device),
            ], dim=1)
        return tokenizer.decode(generated, skip_special_tokens=True).strip()


def inference_few_samples(sample_indices=(2, 7, 11)):
    valid_indices = [index for index in sample_indices if index < len(val_dataset)]
    if not valid_indices:
        raise ValueError("No requested validation index exists.")
    fig, axes = plt.subplots(len(valid_indices), 1, figsize=(13, 4 * len(valid_indices)))
    axes = np.atleast_1d(axes)
    for axis, index in zip(axes, valid_indices):
        features, question, reference = val_dataset[index]
        prediction = greedy_search_single(
            features, question, model, tokenizer, max_length=args.seq_length, device=device
        )
        axis.imshow(features[:64, :128], aspect="auto", cmap="viridis")
        axis.set_title(
            f"Q: {question[:160]}\nReference: {reference[:160]}\nPrediction: {prediction[:160]}",
            fontsize=9,
        )
        axis.set_xlabel("feature dimension")
        axis.set_ylabel("WSI patch token")
    plt.tight_layout()
    plt.show()


inference_few_samples()

# Validation inference: utility sanity check

This is not the final reported test utility. Official-test utility is computed
later from the greedy predictions generated after the clustering lock.

In [ ]:
# Validation-only utility sanity metrics (test remains untouched)
import os
import numpy as np
import random

import torch
import torch.utils.data
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import InterpolationMode
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model
from peft import  TaskType, LoraConfig

from PIL import Image
from tqdm import tqdm
import evaluate
rouge = evaluate.load("rouge")
import time
import math
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


def batch_greedy_search(images, questions, model, tokenizer, max_length, device):
    answers = []
    batch_size = len(questions)

    model.eval()
    with torch.no_grad():
        # Prepare the prompts for the entire batch
        prompt_texts = [f"Question: {q}\nAnswer:" for q in questions]

        # Tokenize the prompts with padding to handle varying lengths
        prompt_inputs = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding='longest',
            truncation=True,
            max_length=max_length - 1,
            add_special_tokens=False
        )

        # Prepare model inputs
        padded_input_ids = torch.zeros((batch_size, max_length), dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros((batch_size, max_length), device=device)

        orig_length = prompt_inputs['input_ids'].size(1)
        padded_input_ids[:, :orig_length] = prompt_inputs['input_ids'].to(device)
        padded_attention_mask[:, :orig_length] = prompt_inputs['attention_mask'].to(device)

        images = images.to(device)

        # Initialize tensors to store generated tokens
        only_answer_ids = torch.empty((batch_size, 0), dtype=torch.long, device=device)

        # Track which sequences have finished generating
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Record each sample length (number of non-eos tokens)
        valid_lengths = padded_attention_mask.sum(dim=1).long()
        batch_indices = torch.arange(batch_size, device=device)

        for _ in range(max_length - orig_length):
            max_valid_lengths = valid_lengths.max().item()

            logits = model(
                image=images,
                qa_inputs_ids=padded_input_ids[:, :max_valid_lengths],
                qa_att_mask=padded_attention_mask[:, :max_valid_lengths]
            )

            last_valid_logits = logits[batch_indices, valid_lengths - 1, :]
            next_token_ids = torch.argmax(last_valid_logits, dim=-1)

            is_eos = next_token_ids == tokenizer.eos_token_id
            finished = finished | is_eos

            padded_input_ids[batch_indices, valid_lengths] = next_token_ids
            padded_attention_mask[batch_indices, valid_lengths] = 1
            valid_lengths += 1

            only_answer_ids = torch.cat(
                [only_answer_ids, next_token_ids.unsqueeze(1)],
                dim=1
            )

            if finished.all():
                break

        # Decode the generated tokens into strings
        generated_ids_cpu = only_answer_ids.cpu().tolist()  # Move to CPU and convert to list for processing
        for i in range(batch_size):
            # Find the first occurrence of eos_token_id to truncate the answer
            try:
                eos_index = generated_ids_cpu[i].index(tokenizer.eos_token_id)
                answer_ids = generated_ids_cpu[i][:eos_index]
            except ValueError:
                # If eos_token_id is not found, use all generated tokens
                answer_ids = generated_ids_cpu[i]

            # Decode the token IDs to a string, skipping special tokens
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
            answers.append(answer)

    return answers

def evaluate_vqa_split(args, data_loader, model, tokenizer, device):
    references = []
    hypotheses = []

    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(tqdm(data_loader), 0):
            images = images.to(device)
            generated_answers = batch_greedy_search(
                images,
                questions,
                model,
                tokenizer,
                max_length=args.seq_length,
                device=device
            )

            references.extend(answers)
            hypotheses.extend(generated_answers)

    return references, hypotheses

def get_nlp_mettics(references, hypotheses):
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor = evaluate.load('meteor')

    # compute HF metrics
    results_bleu = bleu.compute(predictions=hypotheses, references=references)
    results_rouge = rouge.compute(predictions=hypotheses, references=references)
    results_meteor = meteor.compute(predictions=hypotheses, references=references)

    print("HuggingFace Metrics Results:")

    print(f"BLEU-1: {results_bleu['precisions'][0]:.6f}, "
      f"BLEU-2: {results_bleu['precisions'][1]:.6f}, ")

    # print(f"BLEU-4: {results_bleu['bleu']:.6f}")
    print(f"RougeL: {results_rouge['rougeL']:.6f}")
    print(f"Meteor: {results_meteor['meteor']:.6f}")


if __name__ == '__main__':
    # parameters
    random_seed = 42
    seed_everything(random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_dataset = val_dataset
    validation_dataloader = DataLoader(validation_dataset, batch_size=64, shuffle=False, num_workers=4)
    print('Full validation size (test remains untouched):', len(validation_dataset))

    # load weights
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    save_dir = MODEL_CHECKPOINT_PATH
    # save_dir = f'best_model_ca_lr1.pth'
    model.load_state_dict(load_vqa_checkpoint(save_dir, device))
    model.to(device)
    model.eval()

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_references, validation_hypotheses = evaluate_vqa_split(
        args,
        data_loader=validation_dataloader,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    get_nlp_mettics(validation_references, validation_hypotheses)

# Perturbation-Aware Semantic Hallucination Entropy (PA-SHE) for WSI-Bench

## Experimental flow

```mermaid
flowchart TB
    A["Official WSI-Bench training annotations"] --> B["Slide-disjoint train and validation split"]
    B --> C["Train or reuse WSI feature + GPT-2 LoRA model"]
    C --> D["Frozen validation sampling under four conditions"]
    D --> E1["Exact-text clustering"]
    D --> E2["SBERT and BGE cosine grids<br/>0.70, 0.80, 0.90"]
    D --> E3["RoBERTa/DeBERTa mutual-NLI grids<br/>0.35, 0.50, 0.65"]
    E1 --> F["Validation PA-SHE AUROC<br/>ROUGE-L below 0.50"]
    E2 --> F
    E3 --> F
    F --> G["Lock one overall clustering<br/>and one open-ended clustering"]
    H["Official open + closed test annotations"] --> I["Frozen test sampling under four conditions"]
    G --> J["Apply the corresponding locked clustering"]
    I --> J
    J --> K["Overall and open-ended safety"]
    K --> L["Primary label: ROUGE-L below 0.50"]
    K --> M["Sensitivity only: 0.30 and 0.70"]
```

## 1. Configuration and reproducibility

The complete slide-disjoint validation subset created from the official
training annotations is used for clustering selection. The selected official
test annotations are used once for locked evaluation when
`JSE_MAX_EXAMPLES = None`. Sample and feature caches are tied to the checkpoint,
dataset contents, generation settings, split, and candidate configuration.

For a shorter runtime, PA-SHE answers are generated in condition-level batches.
The retained comparison is limited to SE, SNNE, embedding QA-SNNE, VASE, and PA-SHE.

In [ ]:
# Install once if needed:
# !pip install -q pandas scipy scikit-learn sentence-transformers transformers rouge-score seaborn

import os
import hashlib
import json
from pathlib import Path
import random
import re

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from rouge_score import rouge_scorer
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

_jse_max_examples = int(os.environ.get("JSE_MAX_EXAMPLES", "0"))
JSE_MAX_EXAMPLES = None if _jse_max_examples <= 0 else _jse_max_examples
JSE_NUM_SAMPLES = 5
JSE_GENERATION_BATCH_SIZE = int(os.environ.get("JSE_GENERATION_BATCH_SIZE", "20"))
JSE_MAX_NEW_TOKENS = 48
JSE_TEMPERATURE = 1.0
JSE_TOP_P = 0.90
JSE_RANDOM_SEED = 42

# Question-Aligned Semantic Nearest Neighbor Entropy (QA-SNNE).
QA_SNNE_NUM_SAMPLES = int(os.environ.get("QA_SNNE_NUM_SAMPLES", "20"))
QA_SNNE_TEMPERATURE = 1.0
QA_SNNE_TOP_K = 50
QA_SNNE_TOP_P = 0.90
QA_SNNE_BETA = 10.0
QA_SNNE_TAU = 1.0
QA_SNNE_CACHE_SCHEMA_VERSION = 2
QA_SNNE_EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
QA_SNNE_VARIANTS = {
    "Embedding": "qa_snne_embedding",
}
if QA_SNNE_NUM_SAMPLES < 2:
    raise ValueError("QA_SNNE_NUM_SAMPLES must be at least two.")
if JSE_GENERATION_BATCH_SIZE < 1:
    raise ValueError("JSE_GENERATION_BATCH_SIZE must be positive.")

JSE_LABEL_THRESHOLDS = [0.30, 0.50, 0.70]
JSE_PRIMARY_LABEL_THRESHOLD = 0.50

# Predeclared method/threshold grid. Validation selects; test never does.
JSE_SBERT_THRESHOLDS = [0.70, 0.80, 0.90]
JSE_BGE_THRESHOLDS = [0.70, 0.80, 0.90]
JSE_ROBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
JSE_DEBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
JSE_SELECTION_CANDIDATES = (
    ["Exact text"]
    + [f"SBERT@{threshold:.2f}" for threshold in JSE_SBERT_THRESHOLDS]
    + [f"BGE@{threshold:.2f}" for threshold in JSE_BGE_THRESHOLDS]
    + [
        f"RoBERTa-NLI@{threshold:.2f}"
        for threshold in JSE_ROBERTA_NLI_THRESHOLDS
    ]
    + [
        f"DeBERTa-NLI@{threshold:.2f}"
        for threshold in JSE_DEBERTA_NLI_THRESHOLDS
    ]
)

JSE_SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
JSE_BGE_MODEL = "BAAI/bge-small-en-v1.5"
JSE_ROBERTA_NLI_MODEL = "roberta-large-mnli"
JSE_DEBERTA_NLI_MODEL = "microsoft/deberta-large-mnli"
JSE_MODEL_BATCH_SIZE = 64
JSE_CACHE_SCHEMA_VERSION = 4
JSE_FEATURE_SCHEMA_VERSION = 7

required = [
    "model", "tokenizer", "device", "train_dataset", "val_dataset",
    "test_dataset", "MODEL_CHECKPOINT_PATH",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the original notebook through model loading/evaluation first. Missing: "
        + ", ".join(missing)
    )

JSE_CHECKPOINT_PATH = Path(MODEL_CHECKPOINT_PATH).resolve()
if not JSE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {JSE_CHECKPOINT_PATH}")
JSE_CACHE_DIR = (
    JSE_CHECKPOINT_PATH.parent / "pa_she_cache_wsibench"
)
JSE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 2. WSI-feature perturbations and sampled generation

The frozen VQA model receives four conditions for the same slide/question:

- **original:** unmodified WSI patch features and original question;
- **weak:** low-amplitude feature noise representing a mild visual change;
- **distorted:** stronger feature noise plus patch-token replacement, representing
  partial loss/corruption of WSI evidence;
- **paraphrase:** original features with a meaning-preserving question rewrite.

These are embedding-space perturbations because WSI-Bench annotations reference
precomputed WSI patch features rather than ordinary RGB images.

In [ ]:
def jse_normalize(text):
    text = re.sub(r"[^a-z0-9%.\-\s]", " ", str(text).lower().strip())
    return re.sub(r"\s+", " ", text).strip() or "<empty>"


def jse_paraphrase(question):
    q = str(question).strip().rstrip("?")
    lower = q.lower()
    rules = [
        (r"^what does (?:this|the) image show$", "What is shown in the image?"),
        (r"^what is shown in (?:this|the) image$", "What does the image show?"),
        (r"^is there (.+)$", r"Does the image show \1?"),
        (r"^where is (.+) located$", r"What is the location of \1?"),
        (r"^how many (.+) are (?:there|present)$", r"What number of \1 are present?"),
        (r"^what is present$", "What finding is present?"),
    ]
    for pattern, replacement in rules:
        if re.match(pattern, lower):
            return re.sub(pattern, replacement, lower).strip().capitalize()
    return f"Considering the pathology image, {q[0].lower() + q[1:]}?"


def jse_feature_scale(features):
    scale = features.std(dim=0, keepdim=True, unbiased=False)
    return scale.clamp_min(1e-6)


def jse_weak_image(image):
    x = image.detach().cpu().float().clone()
    noise = 0.015 * jse_feature_scale(x) * torch.randn_like(x)
    return x + noise


def jse_distorted_image(image):
    x = image.detach().cpu().float().clone()
    scale = jse_feature_scale(x)
    x = x + 0.08 * scale * torch.randn_like(x)
    replacement_count = max(1, int(round(0.20 * x.shape[0])))
    replacement_indices = torch.randperm(x.shape[0])[:replacement_count]
    replacement = x.mean(dim=0, keepdim=True)
    x[replacement_indices] = replacement
    return x


def jse_top_p_filter(logits, top_p):
    if top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    filtered = torch.full_like(logits, float("-inf"))
    return filtered.scatter(-1, sorted_indices, sorted_logits)


@torch.inference_mode()
def jse_generate_batch(images, question, do_sample=True):
    """Generate one answer per WSI feature tensor in a single model batch."""
    if not images:
        return []
    prompt = f"Question: {question}\nAnswer:"
    prompt_limit = max(8, int(args.seq_length) - JSE_MAX_NEW_TOKENS)
    encoded = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False,
        truncation=True, max_length=prompt_limit,
    )
    batch_size = len(images)
    input_ids = encoded["input_ids"].to(device).repeat(batch_size, 1)
    attention = encoded["attention_mask"].to(device).repeat(batch_size, 1)
    image_batch = torch.stack([image.detach().cpu().float() for image in images]).to(device)
    generated_ids = [[] for _ in range(batch_size)]
    sequence_logprobs = [[] for _ in range(batch_size)]
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(JSE_MAX_NEW_TOKENS):
        scaled = model(
            image=image_batch, qa_inputs_ids=input_ids, qa_att_mask=attention,
        )[:, -1, :] / JSE_TEMPERATURE
        log_probs = torch.log_softmax(scaled, dim=-1)
        if do_sample:
            sampling_logits = jse_top_p_filter(scaled, JSE_TOP_P)
            next_ids = torch.distributions.Categorical(logits=sampling_logits).sample()
        else:
            next_ids = torch.argmax(scaled, dim=-1)
        next_ids = torch.where(
            finished, torch.full_like(next_ids, tokenizer.eos_token_id), next_ids
        )
        selected_logps = log_probs.gather(1, next_ids[:, None]).squeeze(1)
        for sample_index, token_id in enumerate(next_ids.detach().cpu().tolist()):
            if not finished[sample_index] and token_id != tokenizer.eos_token_id:
                generated_ids[sample_index].append(token_id)
                sequence_logprobs[sample_index].append(
                    float(selected_logps[sample_index].item())
                )
        finished = finished | (next_ids == tokenizer.eos_token_id)
        input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
        attention = torch.cat([
            attention,
            torch.ones((batch_size, 1), dtype=attention.dtype, device=device),
        ], dim=1)
        if bool(finished.all()):
            break

    return [{
        "answer": tokenizer.decode(token_ids, skip_special_tokens=True).strip(),
        "sequence_logprob": float(np.sum(logps)) if logps else -50.0,
    } for token_ids, logps in zip(generated_ids, sequence_logprobs)]


def jse_generate_one(image, question, do_sample=True):
    return jse_generate_batch([image], question, do_sample=do_sample)[0]


@torch.inference_mode()
def qa_snne_generate_samples(image, question, num_samples):
    """Generate QA-SNNE samples from the original input in one batch."""
    prompt = f"Question: {question}\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(device).repeat(num_samples, 1)
    attention = encoded["attention_mask"].to(device).repeat(num_samples, 1)
    image_batch = image.unsqueeze(0).to(device).repeat(num_samples, 1, 1)
    generated_ids = [[] for _ in range(num_samples)]
    finished = torch.zeros(num_samples, dtype=torch.bool, device=device)
    for _ in range(JSE_MAX_NEW_TOKENS):
        next_logits = model(
            image=image_batch, qa_inputs_ids=input_ids, qa_att_mask=attention
        )[:, -1, :] / QA_SNNE_TEMPERATURE
        top_k = min(QA_SNNE_TOP_K, next_logits.shape[-1])
        if top_k > 0:
            kth = torch.topk(next_logits, top_k, dim=-1).values[:, -1:]
            next_logits = next_logits.masked_fill(
                next_logits < kth, float("-inf")
            )
        next_logits = jse_top_p_filter(next_logits, QA_SNNE_TOP_P)
        next_ids = torch.distributions.Categorical(logits=next_logits).sample()
        next_ids = torch.where(
            finished, torch.full_like(next_ids, tokenizer.eos_token_id), next_ids
        )
        for sample_index, token_id in enumerate(next_ids.detach().cpu().tolist()):
            if not finished[sample_index] and token_id != tokenizer.eos_token_id:
                generated_ids[sample_index].append(token_id)
        finished = finished | (next_ids == tokenizer.eos_token_id)
        input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
        attention = torch.cat([
            attention,
            torch.ones((num_samples, 1), dtype=attention.dtype, device=device),
        ], dim=1)
        if bool(finished.all()):
            break
    return [
        tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        for token_ids in generated_ids
    ]


def jse_collect_example(
    dataset,
    dataset_index,
    split_name,
):
    image, question, reference = dataset[dataset_index]
    paraphrase = jse_paraphrase(question)
    condition_inputs = {
        "original": ([image.clone() for _ in range(JSE_NUM_SAMPLES)], question),
        "weak": ([jse_weak_image(image) for _ in range(JSE_NUM_SAMPLES)], question),
        "distorted": ([jse_distorted_image(image) for _ in range(JSE_NUM_SAMPLES)], question),
        "paraphrase": ([image.clone() for _ in range(JSE_NUM_SAMPLES)], paraphrase),
    }
    records = []
    for condition, (condition_images, condition_question) in condition_inputs.items():
        for start in range(0, len(condition_images), JSE_GENERATION_BATCH_SIZE):
            batch_records = jse_generate_batch(
                condition_images[start:start + JSE_GENERATION_BATCH_SIZE],
                condition_question,
                do_sample=True,
            )
            for record in batch_records:
                record["condition"] = condition
                records.append(record)
    greedy = jse_generate_one(
        image,
        question,
        do_sample=False,
    )["answer"]
    return {
        "cache_schema_version": JSE_CACHE_SCHEMA_VERSION,
        "split": str(split_name),
        "dataset_index": int(dataset_index),
        "question": question,
        "paraphrase": paraphrase,
        "reference": str(reference),
        "answer_type": str(dataset.dataset[dataset_index].get("answer_type", "Open-ended")),
        "slide_id": str(dataset.dataset[dataset_index].get("slide_id", dataset_index)),
        "greedy": greedy,
        "records": records,
    }

## 3. Dataset-specific clustering backends

WSI-Bench validation compares Exact text with SBERT/BGE cosine thresholds
`0.70`, `0.80`, and `0.90`, plus bidirectional RoBERTa/DeBERTa-NLI
thresholds `0.35`, `0.50`, and `0.65`. Each method's pairwise score matrix is
calculated once per example and reused across thresholds. Official test
constructs only the separately validation-locked overall/open configurations.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

jse_sbert = SentenceTransformer(JSE_SBERT_MODEL, device=str(device))
jse_bge = SentenceTransformer(JSE_BGE_MODEL, device=str(device))


def jse_load_nli(model_name):
    tokenizer_nli = AutoTokenizer.from_pretrained(model_name)
    model_nli = AutoModelForSequenceClassification.from_pretrained(
        model_name
    ).to(device).eval()
    entailment_id = next(
        (
            int(index)
            for index, label in model_nli.config.id2label.items()
            if "entail" in str(label).lower()
        ),
        2,
    )
    return tokenizer_nli, model_nli, entailment_id


jse_roberta_tok, jse_roberta, jse_roberta_entail = jse_load_nli(
    JSE_ROBERTA_NLI_MODEL
)
jse_deberta_tok, jse_deberta, jse_deberta_entail = jse_load_nli(
    JSE_DEBERTA_NLI_MODEL
)


def jse_unique_answers(answers):
    normalized = [jse_normalize(answer) for answer in answers]
    unique = list(dict.fromkeys(normalized))
    return normalized, unique


def jse_exact_clusters(answers):
    normalized, unique = jse_unique_answers(answers)
    mapping = {answer: index for index, answer in enumerate(unique)}
    return [mapping[answer] for answer in normalized]


def jse_embedding_cache(answers, encoder):
    normalized, unique = jse_unique_answers(answers)
    embeddings = encoder.encode(
        unique,
        normalize_embeddings=True,
        batch_size=JSE_MODEL_BATCH_SIZE,
    )
    matrix = np.asarray(embeddings) @ np.asarray(embeddings).T
    return normalized, unique, matrix


@torch.inference_mode()
def jse_nli_cache(answers, tokenizer_nli, model_nli, entailment_id):
    normalized, unique = jse_unique_answers(answers)
    scores = np.eye(len(unique), dtype=float)
    pairs = [
        (i, j)
        for i in range(len(unique))
        for j in range(len(unique))
        if i != j
    ]
    for start in range(0, len(pairs), JSE_MODEL_BATCH_SIZE):
        batch = pairs[start:start + JSE_MODEL_BATCH_SIZE]
        encoded = tokenizer_nli(
            [unique[i] for i, _ in batch],
            [unique[j] for _, j in batch],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        probabilities = torch.softmax(
            model_nli(**encoded).logits,
            dim=-1,
        )[:, entailment_id]
        for (i, j), value in zip(batch, probabilities.cpu().tolist()):
            scores[i, j] = value
    return normalized, unique, scores


def jse_clusters_from_similarity(cache, threshold, bidirectional=False):
    normalized, unique, matrix = cache
    representatives = []
    unique_cluster_ids = []
    for i in range(len(unique)):
        assigned = None
        for cluster_id, representative in enumerate(representatives):
            forward = matrix[i, representative] >= threshold
            backward = matrix[representative, i] >= threshold
            if forward and (backward if bidirectional else True):
                assigned = cluster_id
                break
        if assigned is None:
            assigned = len(representatives)
            representatives.append(i)
        unique_cluster_ids.append(assigned)
    mapping = dict(zip(unique, unique_cluster_ids))
    return [mapping[item] for item in normalized]
_qa_snne_embedding_encoder = None


def qa_snne_get_embedding_encoder():
    global _qa_snne_embedding_encoder
    if _qa_snne_embedding_encoder is None:
        _qa_snne_embedding_encoder = SentenceTransformer(
            QA_SNNE_EMBEDDING_MODEL, device=str(device)
        )
    return _qa_snne_embedding_encoder


def qa_snne_embedding_alignment(question, answers):
    encoder = qa_snne_get_embedding_encoder()
    embeddings = np.asarray(encoder.encode(
        [str(question)] + [str(answer) for answer in answers],
        normalize_embeddings=True,
        batch_size=JSE_MODEL_BATCH_SIZE,
    ))
    return embeddings[1:] @ embeddings[0]


# Only the embedding-based QA-SNNE comparator is retained.


## 4. Risk definitions

For condition \(k\), sampled sequence log-probabilities are normalised within that condition and accumulated by semantic cluster:

\[
p_k(c)=\frac{\sum_{s\in k,\ z(s)=c}\exp(\ell_s)}
{\sum_{s\in k}\exp(\ell_s)}.
\]

With \(K\) available conditions, PA-SHE uses \(\bar p(c)=K^{-1}\sum_k p_k(c)\) and \(H_{\mathrm{joint}}=-\sum_c\bar p(c)\log\bar p(c)\). SE uses only \(p_{\mathrm{original}}\). VASE is the Jensen–Shannon divergence between weak and distorted condition distributions. SNNE uses pairwise ROUGE-L among 20 original-input samples; QA-SNNE adds question–answer embedding alignment.

In [ ]:
JSE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]


def jse_entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log(p + 1e-12)).sum())


def jse_distribution(records, cluster_ids, condition, cluster_count):
    indices = [i for i, r in enumerate(records) if r["condition"] == condition]
    p = np.zeros(cluster_count, dtype=float)
    if not indices:
        return p
    logps = np.asarray([records[i]["sequence_logprob"] for i in indices])
    weights = np.exp(logps - logps.max())
    weights /= max(weights.sum(), 1e-12)
    for i, weight in zip(indices, weights):
        p[int(cluster_ids[i])] += float(weight)
    return p / max(p.sum(), 1e-12)


def jse_signals(example, all_cluster_ids):
    records = example["records"]
    record_ids = all_cluster_ids[:-1]
    cluster_count = max(all_cluster_ids) + 1
    distributions = {
        condition: jse_distribution(records, record_ids, condition, cluster_count)
        for condition in JSE_CONDITIONS
    }
    available = [p for p in distributions.values() if p.sum() > 0]
    joint = np.mean(available, axis=0)
    p_original = distributions["original"]
    p_weak, p_distorted = distributions["weak"], distributions["distorted"]
    vase = float(jensenshannon(
        p_weak + 1e-12, p_distorted + 1e-12, base=2.0
    ) ** 2)
    return {
        "vase": vase,
        "semantic_entropy": jse_entropy(p_original),
        "joint_se": jse_entropy(joint),
        "cluster_count": int(cluster_count),
    }


jse_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def jse_rouge_l(reference, prediction):
    return float(jse_rouge.score(str(reference), str(prediction))["rougeL"].fmeasure)

def qa_snne_rouge_similarity_matrix(answers):
    answers = [jse_normalize(answer) for answer in answers]
    n = len(answers)
    matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            forward = jse_rouge_l(answers[i], answers[j])
            backward = jse_rouge_l(answers[j], answers[i])
            matrix[i, j] = matrix[j, i] = 0.5 * (forward + backward)
    return matrix


def qa_snne_score(similarity_matrix, alignment_scores=None):
    """Equations (1)-(4) of Carlini et al.; higher means less certain."""
    similarity = np.asarray(similarity_matrix, dtype=np.float64)
    n = similarity.shape[0]
    if similarity.shape != (n, n) or n < 2:
        raise ValueError("SNNE requires a square matrix with at least two answers.")
    if alignment_scores is not None:
        alignment = np.asarray(alignment_scores, dtype=np.float64)
        if alignment.shape != (n,):
            raise ValueError("QA-SNNE alignment scores must match sampled answers.")
        shifted = QA_SNNE_BETA * alignment
        shifted -= shifted.max()
        relevance = np.exp(shifted)
        relevance /= max(relevance.sum(), 1e-12)
        similarity = np.diag(relevance) @ similarity @ np.diag(relevance)

    row_log_sums = []
    for i in range(n):
        values = np.delete(similarity[i], i) / QA_SNNE_TAU
        maximum = float(values.max())
        row_log_sums.append(
            maximum + np.log(np.exp(values - maximum).sum() + 1e-12)
        )
    return float(-np.mean(row_log_sums))


def qa_snne_signals(example, qa_sample_example):
    answers = [str(answer) for answer in qa_sample_example["answers"]]
    if len(answers) != QA_SNNE_NUM_SAMPLES:
        raise ValueError("QA-SNNE sample count does not match configuration.")
    question = str(example["question"])
    similarity = qa_snne_rouge_similarity_matrix(answers)
    embedding_alignment = qa_snne_embedding_alignment(question, answers)
    return {
        "snne": qa_snne_score(similarity),
        "qa_snne_embedding": qa_snne_score(similarity, embedding_alignment),
        "qa_snne_embedding_alignment_mean": float(np.mean(embedding_alignment)),
    }

## 5. Validation sampling and cache

Only validation examples are sampled in this cell. The official test split is
not sampled until section 7 has selected and locked a clustering rule.

In [ ]:
JSE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]
jse_datasets = {
    "validation": val_dataset,
    "test": test_dataset,
}


def jse_dataset_signature(dataset, evaluation_size):
    digest = hashlib.sha256()
    for dataset_index in range(evaluation_size):
        raw = dataset.dataset[dataset_index]
        digest.update(str(dataset_index).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["question"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["answer"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw.get("slide_id", "")).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw.get("answer_type", "")).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def jse_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if JSE_MAX_EXAMPLES is None
        else min(int(JSE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = JSE_CHECKPOINT_PATH.stat()
    cache_configuration = {
        "cache_schema_version": JSE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": jse_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(JSE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "samples_per_condition": JSE_NUM_SAMPLES,
        "maximum_new_tokens": JSE_MAX_NEW_TOKENS,
        "temperature": JSE_TEMPERATURE,
        "top_p": JSE_TOP_P,
        "seed": JSE_RANDOM_SEED,
        "conditions": JSE_CONDITIONS,
    }
    cache_hash = hashlib.sha256(
        json.dumps(cache_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = JSE_CACHE_DIR / f"{split_name}_samples_{cache_hash}.jsonl"

    cached_examples = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as cache_file:
            for line_number, line in enumerate(cache_file, start=1):
                if not line.strip():
                    continue
                try:
                    example = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete cache line {line_number}: {cache_path}")
                    continue
                dataset_index = int(example.get("dataset_index", -1))
                if example.get("cache_schema_version") != JSE_CACHE_SCHEMA_VERSION:
                    raise ValueError("PA-SHE sample-cache schema mismatch.")
                if example.get("split") != split_name:
                    raise ValueError("PA-SHE sample-cache split mismatch.")
                if not 0 <= dataset_index < evaluation_size:
                    raise ValueError("Cached dataset index is outside this run.")
                condition_counts = {
                    condition: sum(
                        record.get("condition") == condition
                        for record in example.get("records", [])
                    )
                    for condition in JSE_CONDITIONS
                }
                if any(
                    count != JSE_NUM_SAMPLES
                    for count in condition_counts.values()
                ):
                    raise ValueError("Cached condition/sample counts do not match.")
                if dataset_index in cached_examples:
                    raise ValueError("Duplicate dataset index in PA-SHE cache.")
                cached_examples[dataset_index] = example

    pending_indices = [
        index for index in range(evaluation_size)
        if index not in cached_examples
    ]
    print({
        "split": split_name,
        "sample_cache": str(cache_path),
        "cached_examples": len(cached_examples),
        "pending_examples": len(pending_indices),
    })

    split_seed_offset = 0 if split_name == "validation" else 10_000_000
    with cache_path.open("a", encoding="utf-8") as cache_file:
        for dataset_index in tqdm(
            pending_indices,
            desc=f"Frozen-VQA sampling: {split_name}",
        ):
            example_seed = JSE_RANDOM_SEED + split_seed_offset + dataset_index * 1009
            random.seed(example_seed)
            np.random.seed(example_seed)
            torch.manual_seed(example_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(example_seed)
            example = jse_collect_example(
                dataset=dataset,
                dataset_index=dataset_index,
                split_name=split_name,
            )
            cache_file.write(json.dumps(example, ensure_ascii=False) + "\n")
            cache_file.flush()
            cached_examples[dataset_index] = example

    return (
        [cached_examples[index] for index in range(evaluation_size)],
        cache_path,
    )


# Selection starts with validation only. Test sampling occurs in section 7,
# after JSE_LOCKED_CLUSTERING has been assigned.
def qa_snne_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if JSE_MAX_EXAMPLES is None
        else min(int(JSE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = JSE_CHECKPOINT_PATH.stat()
    configuration = {
        "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": jse_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(JSE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "num_samples": QA_SNNE_NUM_SAMPLES,
        "maximum_new_tokens": JSE_MAX_NEW_TOKENS,
        "temperature": QA_SNNE_TEMPERATURE,
        "top_k": QA_SNNE_TOP_K,
        "top_p": QA_SNNE_TOP_P,
        "seed": JSE_RANDOM_SEED,
        "input_condition": "original image and original question",
    }
    cache_hash = hashlib.sha256(
        json.dumps(configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = JSE_CACHE_DIR / f"{split_name}_qa_snne_samples_{cache_hash}.jsonl"
    cached = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete QA-SNNE cache line {line_number}")
                    continue
                index = int(record.get("dataset_index", -1))
                if record.get("cache_schema_version") != QA_SNNE_CACHE_SCHEMA_VERSION:
                    raise ValueError("QA-SNNE sample-cache schema mismatch.")
                if record.get("split") != split_name or not 0 <= index < evaluation_size:
                    raise ValueError("QA-SNNE sample-cache split/index mismatch.")
                if len(record.get("answers", [])) != QA_SNNE_NUM_SAMPLES:
                    raise ValueError("QA-SNNE cached sample count mismatch.")
                if index in cached:
                    raise ValueError("Duplicate index in QA-SNNE sample cache.")
                cached[index] = record

    pending = [index for index in range(evaluation_size) if index not in cached]
    print({
        "split": split_name,
        "qa_snne_sample_cache": str(cache_path),
        "cached_examples": len(cached),
        "pending_examples": len(pending),
        "samples_per_example": QA_SNNE_NUM_SAMPLES,
    })
    split_offset = 30_000_000 if split_name == "validation" else 40_000_000
    with cache_path.open("a", encoding="utf-8") as handle:
        for index in tqdm(pending, desc=f"QA-SNNE sampling: {split_name}"):
            seed = JSE_RANDOM_SEED + split_offset + index * 1013
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            image, question, _ = dataset[index]
            record = {
                "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
                "split": split_name,
                "dataset_index": int(index),
                "question": str(question),
                "answers": qa_snne_generate_samples(
                    image, question, QA_SNNE_NUM_SAMPLES
                ),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            cached[index] = record
    return [cached[index] for index in range(evaluation_size)], cache_path

jse_validation_examples, jse_validation_sample_cache_path = jse_collect_split(
    "validation",
    jse_datasets["validation"],
)
jse_examples_by_split = {"validation": jse_validation_examples}
jse_sample_cache_paths = {"validation": jse_validation_sample_cache_path}
qa_snne_validation_examples, qa_snne_validation_sample_cache_path = (
    qa_snne_collect_split("validation", jse_datasets["validation"])
)
qa_snne_examples_by_split = {"validation": qa_snne_validation_examples}
qa_snne_sample_cache_paths = {
    "validation": qa_snne_validation_sample_cache_path
}

print({
    "validation_examples": len(jse_validation_examples),
    "test_sampled_before_selection": False,
})
display(pd.DataFrame([{
    "split": example["split"],
    "index": example["dataset_index"],
    "question": example["question"],
    "reference": example["reference"],
    "greedy": example["greedy"],
} for example in jse_validation_examples[:5]]))

## 6. Validation candidate features

Build validation features for the predeclared 13-candidate grid: Exact text
and three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
The same cache also stores VASE, SNNE, and embedding-based QA-SNNE.
No test feature exists yet.

In [ ]:
def jse_clusters_for_configurations(answers, clustering_configurations):
    clustering_configurations = list(clustering_configurations)
    unsupported = set(clustering_configurations) - set(JSE_SELECTION_CANDIDATES)
    if unsupported:
        raise ValueError(
            "Unsupported WSI-Bench clustering configuration(s): "
            + ", ".join(sorted(unsupported))
        )

    requested = set(clustering_configurations)
    configurations = []
    if "Exact text" in requested:
        configurations.append(("Exact text", jse_exact_clusters(answers)))

    embedding_specs = [
        ("SBERT", JSE_SBERT_THRESHOLDS, jse_sbert),
        ("BGE", JSE_BGE_THRESHOLDS, jse_bge),
    ]
    for method_name, thresholds, encoder in embedding_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = jse_embedding_cache(answers, encoder)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    jse_clusters_from_similarity(
                        score_cache, threshold, bidirectional=False
                    ),
                ))

    nli_specs = [
        (
            "RoBERTa-NLI", JSE_ROBERTA_NLI_THRESHOLDS,
            jse_roberta_tok, jse_roberta, jse_roberta_entail,
        ),
        (
            "DeBERTa-NLI", JSE_DEBERTA_NLI_THRESHOLDS,
            jse_deberta_tok, jse_deberta, jse_deberta_entail,
        ),
    ]
    for method_name, thresholds, tok, mdl, entail_id in nli_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = jse_nli_cache(answers, tok, mdl, entail_id)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    jse_clusters_from_similarity(
                        score_cache, threshold, bidirectional=True
                    ),
                ))
    if {name for name, _ in configurations} != set(clustering_configurations):
        raise ValueError("Failed to construct every requested clustering.")
    return configurations


def jse_build_feature_frame(
    split_name,
    examples,
    sample_cache_path,
    clustering_configurations,
):
    clustering_configurations = list(clustering_configurations)
    feature_configuration = {
        "feature_schema_version": JSE_FEATURE_SCHEMA_VERSION,
        "split": split_name,
        "sample_cache": sample_cache_path.name,
        "clustering_configurations": clustering_configurations,
        "sbert_model": JSE_SBERT_MODEL,
        "sbert_thresholds": JSE_SBERT_THRESHOLDS,
        "bge_model": JSE_BGE_MODEL,
        "bge_thresholds": JSE_BGE_THRESHOLDS,
        "roberta_nli_model": JSE_ROBERTA_NLI_MODEL,
        "roberta_nli_thresholds": JSE_ROBERTA_NLI_THRESHOLDS,
        "deberta_nli_model": JSE_DEBERTA_NLI_MODEL,
        "deberta_nli_thresholds": JSE_DEBERTA_NLI_THRESHOLDS,
        "qa_snne_sample_cache": qa_snne_sample_cache_paths[split_name].name,
        "qa_snne_num_samples": QA_SNNE_NUM_SAMPLES,
        "qa_snne_beta": QA_SNNE_BETA,
        "qa_snne_tau": QA_SNNE_TAU,
        "qa_snne_embedding_model": QA_SNNE_EMBEDDING_MODEL,
    }
    feature_hash = hashlib.sha256(
        json.dumps(feature_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    feature_path = JSE_CACHE_DIR / f"{split_name}_features_{feature_hash}.csv"

    if feature_path.exists():
        frame = pd.read_csv(feature_path, keep_default_na=False)
        expected_rows = len(examples) * len(clustering_configurations)
        required_columns = {
            "split", "dataset_index", "clustering", "rougeL", "reference",
            "prediction", "answer_type", "vase",
            "semantic_entropy", "joint_se", "cluster_count",
            "snne", "qa_snne_embedding",
        }
        if required_columns - set(frame.columns):
            raise ValueError("Cached WSI-Bench features are incomplete.")
        if len(frame) != expected_rows:
            raise ValueError("Cached WSI-Bench feature row count is incorrect.")
        if set(frame["clustering"]) != set(clustering_configurations):
            raise ValueError("Cached WSI-Bench clustering set is incorrect.")
        print(f"Loaded {split_name} semantic features from {feature_path}")
        return frame, feature_path

    feature_rows = []
    for example in tqdm(examples, desc=f"Semantic clustering: {split_name}"):
        answers = [
            record["answer"] for record in example["records"]
        ] + [example["greedy"]]
        qa_sample_example = qa_snne_examples_by_split[split_name][
            int(example["dataset_index"])
        ]
        qa_uncertainty = qa_snne_signals(example, qa_sample_example)
        configurations = jse_clusters_for_configurations(
            answers,
            clustering_configurations,
        )
        rouge_l = jse_rouge_l(example["reference"], example["greedy"])
        answer_type = str(example.get("answer_type", "Open-ended"))
        for clustering, cluster_ids in configurations:
            row = {
                "split": split_name,
                "dataset_index": int(example["dataset_index"]),
                "clustering": clustering,
                "rougeL": rouge_l,
                "reference": example["reference"],
                "prediction": example["greedy"],
                "answer_type": answer_type,
            }
            row.update(jse_signals(example, cluster_ids))
            row.update(qa_uncertainty)
            feature_rows.append(row)

    frame = pd.DataFrame(feature_rows)
    frame.to_csv(feature_path, index=False)
    print(f"Saved {split_name} semantic features to {feature_path}")
    return frame, feature_path


jse_validation_features, jse_validation_feature_cache_path = (
    jse_build_feature_frame(
        "validation",
        jse_validation_examples,
        jse_validation_sample_cache_path,
        JSE_SELECTION_CANDIDATES,
    )
)
display(jse_validation_features.head())
print({
    "dataset": "WSI-Bench",
    "selection_split": "validation",
    "selection_candidates": JSE_SELECTION_CANDIDATES,
    "test_features_built_before_selection": False,
})

## 7. Separate overall/open-ended validation selection and locked test safety

WSI-Bench contains both descriptive open questions and closed questions. One
clustering backend is selected on all validation examples for overall reporting,
and another is selected on open-ended validation examples. Both choices are locked before official-test sampling.
The test split is evaluated once with those locked choices.

In [ ]:
JSE_RISK_COLUMNS = {
    "VASE": "vase",
    "SE": "semantic_entropy",
    "SNNE": "snne",
    "QA-SNNE · Embedding": "qa_snne_embedding",
    "PA-SHE": "joint_se",
}


def jse_safe_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    if labels.shape != scores.shape:
        raise ValueError("Labels and uncertainty scores must align.")
    if not np.isfinite(scores).all():
        raise ValueError("Uncertainty scores contain NaN or infinity.")
    if np.unique(labels).size < 2:
        return np.nan, np.nan
    return roc_auc_score(labels, scores), average_precision_score(labels, scores)


# Select one overall lock and one open-ended lock using validation only.
def jse_select_clustering(validation_features, selection_subset):
    expected_examples = validation_features["dataset_index"].nunique()
    rows = []
    for candidate_order, clustering in enumerate(JSE_SELECTION_CANDIDATES):
        group = validation_features[
            validation_features["clustering"] == clustering
        ].sort_values("dataset_index")
        if len(group) != expected_examples:
            raise ValueError(
                f"{selection_subset} validation is incomplete for {clustering}."
            )
        failures = (
            group["rougeL"].to_numpy() < JSE_PRIMARY_LABEL_THRESHOLD
        ).astype(int)
        auroc, auprc = jse_safe_metrics(failures, group["joint_se"])
        rows.append({
            "selection_split": "slide-disjoint validation",
            "selection_subset": selection_subset,
            "test_used_for_selection": False,
            "candidate_order": candidate_order,
            "label_threshold": JSE_PRIMARY_LABEL_THRESHOLD,
            "clustering": clustering,
            "examples": len(group),
            "failure_prevalence": failures.mean(),
            "validation_AUROC": auroc,
            "validation_AUPRC": auprc,
        })
    selection = pd.DataFrame(rows)
    ranked = selection.sort_values(
        ["validation_AUROC", "validation_AUPRC", "candidate_order"],
        ascending=[False, False, True],
        kind="mergesort",
    )
    if ranked.empty or pd.isna(ranked.iloc[0]["validation_AUROC"]):
        raise ValueError(
            f"Validation labels cannot select clustering for {selection_subset}."
        )
    return selection, str(ranked.iloc[0]["clustering"])

jse_validation_selection, JSE_LOCKED_CLUSTERING = jse_select_clustering(
    jse_validation_features, "All"
)
open_validation_features = jse_validation_features[
    jse_validation_features["answer_type"] == "Open-ended"
].copy()
if open_validation_features.empty:
    raise ValueError("WSI-Bench validation contains no open-ended examples.")
jse_open_validation_selection, JSE_OPEN_LOCKED_CLUSTERING = (
    jse_select_clustering(open_validation_features, "Open-ended")
)

# Select the QA-SNNE alignment variant independently on validation only.
qa_validation_base = jse_validation_features[
    jse_validation_features["clustering"] == JSE_SELECTION_CANDIDATES[0]
].sort_values("dataset_index")
qa_failures = (
    qa_validation_base["rougeL"].to_numpy() < JSE_PRIMARY_LABEL_THRESHOLD
).astype(int)
qa_selection_rows = []
for variant_order, (variant, column) in enumerate(QA_SNNE_VARIANTS.items()):
    auroc, auprc = jse_safe_metrics(qa_failures, qa_validation_base[column])
    qa_selection_rows.append({
        "selection_split": "validation",
        "selection_subset": "All",
        "test_used_for_selection": False,
        "variant_order": variant_order,
        "label_threshold": JSE_PRIMARY_LABEL_THRESHOLD,
        "variant": variant,
        "column": column,
        "examples": len(qa_validation_base),
        "failure_prevalence": qa_failures.mean(),
        "validation_AUROC": auroc,
        "validation_AUPRC": auprc,
    })
qa_snne_validation_selection = pd.DataFrame(qa_selection_rows)
qa_ranked = qa_snne_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "variant_order"],
    ascending=[False, False, True],
    kind="mergesort",
)
if qa_ranked.empty or pd.isna(qa_ranked.iloc[0]["validation_AUROC"]):
    raise ValueError("Validation labels cannot select a QA-SNNE variant.")
QA_SNNE_LOCKED_VARIANT = str(qa_ranked.iloc[0]["variant"])
QA_SNNE_LOCKED_COLUMN = str(qa_ranked.iloc[0]["column"])
QA_SNNE_LOCKED_METHOD = f"QA-SNNE · {QA_SNNE_LOCKED_VARIANT}"

selection_path = JSE_CACHE_DIR / "validation_selected_clustering.json"
selection_payload = {
    "dataset": "WSI-Bench",
    "selection_split": "slide-disjoint validation",
    "test_used_for_selection": False,
    "primary_label_definition": "ROUGE-L < 0.50",
    "selection_metric": "AUROC; AUPRC tie-breaker; candidate order final tie-breaker",
    "eligible_candidates": JSE_SELECTION_CANDIDATES,
    "overall_locked_clustering": JSE_LOCKED_CLUSTERING,
    "open_ended_locked_clustering": JSE_OPEN_LOCKED_CLUSTERING,
    "qa_snne_locked_variant": QA_SNNE_LOCKED_VARIANT,
    "qa_snne_locked_column": QA_SNNE_LOCKED_COLUMN,
    "qa_snne_selection_metric": (
        "AUROC; AUPRC tie-breaker; variant order final tie-breaker"
    ),
    "qa_snne_candidates": qa_snne_validation_selection.drop(
        columns="variant_order"
    ).to_dict(orient="records"),
    "overall_candidates": jse_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
    "open_ended_candidates": jse_open_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
}
selection_path.write_text(
    json.dumps(selection_payload, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
jse_validation_selection.to_csv(
    JSE_CACHE_DIR / "validation_clustering_selection.csv", index=False
)
jse_open_validation_selection.to_csv(
    JSE_CACHE_DIR / "validation_open_ended_clustering_selection.csv",
    index=False,
)
qa_snne_validation_selection.to_csv(
    JSE_CACHE_DIR / "validation_qa_snne_selection.csv",
    index=False,
)

# Only now sample official test and build the union of the two locked rules.
qa_snne_test_examples, qa_snne_test_sample_cache_path = (
    qa_snne_collect_split("test", jse_datasets["test"])
)
qa_snne_examples_by_split["test"] = qa_snne_test_examples
qa_snne_sample_cache_paths["test"] = qa_snne_test_sample_cache_path

jse_test_examples, jse_test_sample_cache_path = jse_collect_split(
    "test", jse_datasets["test"]
)
jse_examples_by_split["test"] = jse_test_examples
jse_sample_cache_paths["test"] = jse_test_sample_cache_path
locked_configuration_union = list(dict.fromkeys([
    JSE_LOCKED_CLUSTERING,
    JSE_OPEN_LOCKED_CLUSTERING,
]))
jse_test_features, jse_test_feature_cache_path = jse_build_feature_frame(
    "test",
    jse_test_examples,
    jse_test_sample_cache_path,
    locked_configuration_union,
)
jse_locked_test_features = jse_test_features[
    jse_test_features["clustering"] == JSE_LOCKED_CLUSTERING
].sort_values("dataset_index").copy()
if len(jse_locked_test_features) != len(jse_test_examples):
    raise ValueError("Overall lock must cover every official-test question.")
if jse_locked_test_features["dataset_index"].duplicated().any():
    raise ValueError("Locked WSI-Bench test features contain duplicate questions.")
jse_open_locked_test_features = jse_test_features[
    (jse_test_features["clustering"] == JSE_OPEN_LOCKED_CLUSTERING)
    & (jse_test_features["answer_type"] == "Open-ended")
].sort_values("dataset_index").copy()
if jse_open_locked_test_features.empty:
    raise ValueError("WSI-Bench test contains no open-ended examples.")
if jse_open_locked_test_features["dataset_index"].duplicated().any():
    raise ValueError("Open-ended locked WSI-Bench features contain duplicates.")
jse_features = jse_locked_test_features

evaluation_subsets = {
    "All": jse_locked_test_features,
    "Closed-ended": jse_locked_test_features[
        jse_locked_test_features["answer_type"] == "Closed-ended"
    ],
    "Open-ended": jse_open_locked_test_features,
}

jse_result_rows = []
for evaluation_subset, subset_features in evaluation_subsets.items():
    subset_clusterings = subset_features["clustering"].unique()
    if len(subset_clusterings) != 1:
        raise ValueError(
            f"{evaluation_subset} must use exactly one locked clustering."
        )
    subset_locked_clustering = str(subset_clusterings[0])
    for label_threshold in JSE_LABEL_THRESHOLDS:
        failures = (
            subset_features["rougeL"].to_numpy() < label_threshold
        ).astype(int)
        for method, column in JSE_RISK_COLUMNS.items():
            auroc, auprc = jse_safe_metrics(failures, subset_features[column])
            jse_result_rows.append({
                "evaluation_split": "official test",
                "evaluation_subset": evaluation_subset,
                "examples": len(subset_features),
                "label_threshold": label_threshold,
                "failure_prevalence": failures.mean() if len(failures) else np.nan,
                "clustering": subset_locked_clustering,
                "method": method,
                "AUROC": auroc,
                "AUPRC": auprc,
            })

jse_results = pd.DataFrame(jse_result_rows)
jse_primary_results = jse_results[
    (jse_results["evaluation_subset"] == "All")
    & np.isclose(jse_results["label_threshold"], JSE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
jse_label_sensitivity = jse_results[
    (jse_results["evaluation_subset"] == "All")
    & (jse_results["method"] == "PA-SHE")
].sort_values("label_threshold")
jse_open_ended_results = jse_results[
    jse_results["evaluation_subset"] == "Open-ended"
].copy()
jse_open_ended_primary_results = jse_open_ended_results[
    np.isclose(jse_open_ended_results["label_threshold"], JSE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
jse_open_ended_label_sensitivity = jse_open_ended_results[
    jse_open_ended_results["method"] == "PA-SHE"
].sort_values("label_threshold")

jse_results.to_csv(JSE_CACHE_DIR / "locked_test_safety_results.csv", index=False)
jse_open_ended_results.to_csv(
    JSE_CACHE_DIR / "locked_test_open_ended_safety_results.csv", index=False
)

print("VALIDATION-ONLY QA-SNNE variant selection:")
display(qa_snne_validation_selection.drop(columns="variant_order").round(4))
print("Locked QA-SNNE variant before test sampling:", QA_SNNE_LOCKED_VARIANT)
print("VALIDATION-ONLY WSI-Bench clustering selection:")
display(jse_validation_selection.drop(columns="candidate_order").round(4))
print("VALIDATION-ONLY open-ended clustering selection:")
display(
    jse_open_validation_selection.drop(columns="candidate_order").round(4)
)
print("Overall lock before official-test sampling:", JSE_LOCKED_CLUSTERING)
print("Open-ended lock before official-test sampling:", JSE_OPEN_LOCKED_CLUSTERING)
print("OFFICIAL TEST primary results: failure = ROUGE-L < 0.50")
display(jse_primary_results.round(4))
print("OFFICIAL TEST open-ended primary results")
display(jse_open_ended_primary_results.round(4))
print("PA-SHE test label sensitivity; clustering remains locked")
display(jse_label_sensitivity.round(4))
print("Open-ended PA-SHE label sensitivity; clustering remains locked")
display(jse_open_ended_label_sensitivity.round(4))
print("Saved validation selection protocol:", selection_path)

## 8. Validation diagnostic and locked test comparisons

The first plot is the validation-only clustering selection. All subsequent
plots use the single locked clustering on official test.

In [ ]:
# Validation-only overall and open-ended clustering selections.
validation_plots = [
    ("Overall validation", jse_validation_selection),
    ("Open-ended validation", jse_open_validation_selection),
]
fig, axes = plt.subplots(2, 2, figsize=(22, 11))
for row_index, (subset_label, selection_frame) in enumerate(validation_plots):
    validation_plot = selection_frame.sort_values("candidate_order")
    axes[row_index, 0].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUROC"],
        color="#4c78a8",
    )
    axes[row_index, 1].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUPRC"],
        color="#f58518",
    )
    axes[row_index, 0].set_title(f"{subset_label}: PA-SHE AUROC")
    axes[row_index, 1].set_title(f"{subset_label}: PA-SHE AUPRC")
    for axis in axes[row_index]:
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45)
        axis.grid(axis="y", alpha=0.25)
fig.suptitle("WSI-Bench validation locks: ROUGE-L < 0.50")
plt.tight_layout()
plt.show()

# Official-test label sensitivity; overall clustering stays locked.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    jse_label_sensitivity["label_threshold"],
    jse_label_sensitivity["AUROC"], marker="o"
)
axes[1].plot(
    jse_label_sensitivity["label_threshold"],
    jse_label_sensitivity["AUPRC"], marker="o", color="#f58518"
)
axes[0].set_title("Official test AUROC sensitivity")
axes[1].set_title("Official test AUPRC sensitivity")
for axis in axes:
    axis.set_xlabel("ROUGE-L failure-label threshold")
    axis.set_ylim(0, 1)
    axis.set_xticks(JSE_LABEL_THRESHOLDS)
    axis.grid(alpha=0.25)
fig.suptitle(f"PA-SHE with locked clustering: {JSE_LOCKED_CLUSTERING}")
plt.tight_layout()
plt.show()

primary_comparison = jse_primary_results.sort_values("AUROC")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(primary_comparison["method"], primary_comparison["AUROC"])
axes[1].barh(
    primary_comparison["method"], primary_comparison["AUPRC"], color="#d17a22"
)
axes[0].set_title("Official test AUROC")
axes[1].set_title("Official test AUPRC")
for axis in axes:
    axis.set_xlim(0, 1)
    axis.grid(axis="x", alpha=0.25)
fig.suptitle(
    f"Validation-locked WSI-Bench clustering; "
    f"failure = ROUGE-L < {JSE_PRIMARY_LABEL_THRESHOLD:.2f}"
)
plt.tight_layout()
plt.show()

## 9. Locked selective prediction and audit

These analyses use only official-test features from the validation-locked
clustering configuration.

In [ ]:
primary_features = jse_locked_test_features.copy()
rejection_fractions = np.linspace(0, 0.50, 11)
curve_rows = []

for method, column in JSE_RISK_COLUMNS.items():
    ordered = primary_features.sort_values(column, ascending=True)
    for fraction in rejection_fractions:
        retained_count = max(1, int(round(len(ordered) * (1.0 - fraction))))
        retained = ordered.iloc[:retained_count]
        curve_rows.append({
            "method": method,
            "rejected_fraction": fraction,
            "retained_ROUGE-L": retained["rougeL"].mean(),
        })

jse_rejection = pd.DataFrame(curve_rows)
plt.figure(figsize=(11, 6))
for method, group in jse_rejection.groupby("method", sort=False):
    plt.plot(
        group["rejected_fraction"],
        group["retained_ROUGE-L"],
        marker="o",
        label=method,
    )
plt.xlabel("Fraction rejected as high risk")
plt.ylabel("Mean ROUGE-L among retained answers")
plt.title(f"Official test selective prediction: locked {JSE_LOCKED_CLUSTERING}")
plt.grid(alpha=0.25)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

audit_columns = [
    "dataset_index", "answer_type", "reference", "prediction", "rougeL",
    "joint_se", "semantic_entropy",
]
print("Highest PA-SHE official-test cases")
display(primary_features.nlargest(10, "joint_se")[audit_columns].round(4))
print("Low PA-SHE official-test failures")
display(
    primary_features[
        primary_features["rougeL"] < JSE_PRIMARY_LABEL_THRESHOLD
    ].nsmallest(10, "joint_se")[audit_columns].round(4)
)

## 10. Reporting checklist

- Report the 13 WSI-Bench candidates: Exact plus three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
- State that validation is a deterministic slide-level split from the official
  training annotations; it is not an official WSI-Bench validation file.
- Report selection at `ROUGE-L < 0.50` using validation AUROC, AUPRC tie-break,
  then candidate-order tie-break.
- Report the separately locked overall and open-ended clustering rules.
- Evaluate the official test annotations only after both rules are locked.
- Treat label thresholds `0.30` and `0.70` as sensitivity analyses.
- Report overall and open-ended test AUROC/AUPRC separately.

## 11. Main comparison table

The table uses separate overall and open-ended WSI-Bench clustering selected on
validation and locked before test sampling. Utility is shared because all risk methods evaluate the same
deployed greedy predictions. Safety is shown overall and for open-ended
questions, where normalized references other than exact `yes` or `no` are
classified as open-ended.

In [ ]:
# Final comparison from separately locked overall/open-ended configurations.
import evaluate
from IPython.display import display

comparison_required = [
    "jse_results", "jse_open_ended_results", "jse_locked_test_features",
    "JSE_LOCKED_CLUSTERING", "JSE_OPEN_LOCKED_CLUSTERING",
    "JSE_PRIMARY_LABEL_THRESHOLD",
    "QA_SNNE_LOCKED_VARIANT", "QA_SNNE_LOCKED_METHOD",
]
comparison_missing = [
    name for name in comparison_required if name not in globals()
]
if comparison_missing:
    raise RuntimeError(
        "Run PA-SHE sections 5-7 first. Missing: "
        + ", ".join(comparison_missing)
    )

utility_source = jse_locked_test_features.sort_values(
    "dataset_index"
).drop_duplicates("dataset_index")
references = utility_source["reference"].astype(str).tolist()
predictions = utility_source["prediction"].astype(str).tolist()
comparison_utility = {
    "BLEU": 100.0 * float(evaluate.load("bleu").compute(
        predictions=predictions, references=references
    )["bleu"]),
    "ROUGE-L": 100.0 * float(evaluate.load("rouge").compute(
        predictions=predictions, references=references
    )["rougeL"]),
    "METEOR": 100.0 * float(evaluate.load("meteor").compute(
        predictions=predictions, references=references
    )["meteor"]),
}

overall_safety = jse_results[
    (jse_results["evaluation_subset"] == "All")
    & np.isclose(jse_results["label_threshold"], JSE_PRIMARY_LABEL_THRESHOLD)
].copy()
open_safety = jse_open_ended_results[
    np.isclose(
        jse_open_ended_results["label_threshold"],
        JSE_PRIMARY_LABEL_THRESHOLD,
    )
].copy()


def comparison_safety(frame, method):
    match = frame[frame["method"] == method]
    if len(match) != 1:
        raise ValueError(f"Expected one locked result for {method}; found {len(match)}")
    row = match.iloc[0]
    return 100.0 * float(row["AUROC"]), 100.0 * float(row["AUPRC"])


locked_variant = (
    f"Overall={JSE_LOCKED_CLUSTERING}; "
    f"Open-ended={JSE_OPEN_LOCKED_CLUSTERING}"
)
specification = [
    ("Semantic entropy", locked_variant, "SE"),
    ("Semantic nearest-neighbour entropy", f"ROUGE-L · n={QA_SNNE_NUM_SAMPLES}", "SNNE"),
    ("Visual stability", locked_variant, "VASE"),
    ("Perturbation-Aware Semantic Hallucination Entropy (PA-SHE)", locked_variant, "PA-SHE"),
]
for qa_variant in QA_SNNE_VARIANTS:
    selected_suffix = (
        " · validation-selected"
        if qa_variant == QA_SNNE_LOCKED_VARIANT else ""
    )
    specification.append((
        "Question-aligned SNNE",
        f"{qa_variant}{selected_suffix} · beta={QA_SNNE_BETA:g}",
        f"QA-SNNE · {qa_variant}",
    ))

rows = []
for family, variant, method in specification:
    overall_auroc, overall_auprc = comparison_safety(overall_safety, method)
    open_auroc, open_auprc = comparison_safety(open_safety, method)
    rows.append({
        "Uncertainty method": family,
        "Variant / clustering": variant,
        ("Utility", "BLEU"): comparison_utility["BLEU"],
        ("Utility", "ROUGE-L"): comparison_utility["ROUGE-L"],
        ("Utility", "METEOR"): comparison_utility["METEOR"],
        ("Overall safety", "AUROC"): overall_auroc,
        ("Overall safety", "AUPRC"): overall_auprc,
        ("Open-ended safety", "AUROC"): open_auroc,
        ("Open-ended safety", "AUPRC"): open_auprc,
    })

comparison_df = pd.DataFrame(rows).set_index([
    "Uncertainty method", "Variant / clustering"
])
comparison_df.columns = pd.MultiIndex.from_tuples(
    comparison_df.columns,
    names=["Evaluation dimension", "Metric"],
)
safety_columns = [
    ("Overall safety", "AUROC"), ("Overall safety", "AUPRC"),
    ("Open-ended safety", "AUROC"), ("Open-ended safety", "AUPRC"),
]
safety_maxima = {column: comparison_df[column].max() for column in safety_columns}


def highlight_maximum(value, column):
    maximum = safety_maxima.get(column)
    if maximum is not None and pd.notna(value) and np.isclose(value, maximum):
        return "font-weight: 700; background-color: #e8f1fb;"
    return ""


comparison_styler = (
    comparison_df.style
    .format("{:.2f}", na_rep="—")
    .apply(
        lambda series: [highlight_maximum(value, series.name) for value in series],
        axis=0,
    )
    .set_caption(
        "WSI-Bench: Separately Validation-Locked Overall and "
        "Open-Ended Clustering"
    )
    .set_table_styles([
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "18px"),
            ("font-weight", "700"), ("text-align", "left"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "#f5f5f5"), ("border", "1px solid #aaa"),
            ("padding", "8px"), ("text-align", "center"),
        ]},
        {"selector": "td", "props": [
            ("border", "1px solid #b5b5b5"), ("padding", "8px"),
            ("text-align", "center"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"), ("font-size", "13px"),
            ("width", "100%"),
        ]},
    ])
)
display(comparison_styler)
comparison_df.to_csv("WSIBench_PA_SHE_main_comparison_table.csv")
with open(
    "WSIBench_PA_SHE_main_comparison_table.html",
    "w",
    encoding="utf-8",
) as comparison_file:
    comparison_file.write(comparison_styler.to_html())
print("Overall locked clustering:", JSE_LOCKED_CLUSTERING)
print("Open-ended locked clustering:", JSE_OPEN_LOCKED_CLUSTERING)
print("Saved WSIBench_PA_SHE_main_comparison_table.csv/html")

### Reading the table

- Utility scores describe the same frozen WSI-Bench answer model and therefore
  repeat across uncertainty methods.
- Overall safety uses all selected official-test questions; open-ended safety
  uses records labelled `Open-ended` by the annotation normalizer.
- AUROC and AUPRC detect failures defined by `ROUGE-L < 0.50`.
- Semantic rows use the separately locked overall/open-ended clustering; no
  clustering or label threshold is selected on official-test outcomes.
- Bold cells indicate the best displayed safety ranking only.